In [1]:
# Kill all processes on the GPU
!fuser -v /dev/nvidia* -k

In [2]:
# Check GPU status
!nvidia-smis

/bin/bash: line 1: nvidia-smis: command not found


In [3]:
# For Qwen2.5-VL
# # %%capture
# import os, re
# if "COLAB_" not in "".join(os.environ.keys()):
#     %pip install unsloth  # Do this in local & cloud setups
# else:
#     import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
#     xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
#     %pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
#     %pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
# %pip install transformers==4.56.2
# %pip install --no-deps trl==0.22.2

# For Qwen3-VL
# %%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    %pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    %pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    %pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
%pip install transformers==4.57.1
%pip install --no-deps trl==0.22.2

# Configuration

In [4]:
SEED = 42

# Model configuration
SFT_LORA_ID = 'alxxtexxr/Qwen3-VL-8B-Instruct-with-codebook-SFT-LoRA-v20260401082523'

# Data configuration
RL_DATA_ID = 'alxxtexxr/ViRL1.25K'

In [5]:
from datetime import datetime

# Resume training configuration
RESUME_MODEL_ID = None
resume_from_checkpoint = bool(RESUME_MODEL_ID)
if resume_from_checkpoint:
    # hub_model_id = RESUME_MODEL_ID
    # project_name = hub_model_id.split('/')[-1]
    # model_name = project_name

    # from huggingface_hub import snapshot_download
    # snapshot_download(repo_id=hub_model_id, local_dir=model_name)
    
    raise NotImplementedError("Resuming from checkpoint is not implemented yet!")
else:
    # model_name = 'unsloth/Meta-Llama-3.1-8B'
    user_name, model_name = SFT_LORA_ID.split('/')
    model_name = model_name.split('-SFT-LoRA')[0]  # Extract base model name
    project_name = f'{model_name}-GRPO-LoRA-v{datetime.now().strftime("%Y%m%d%H%M%S")}'
    hub_model_id = f'{user_name}/{project_name}'
print("Resume from checkpoint:", resume_from_checkpoint)
print("Project name:", project_name)
print("Hugging Face model ID:", hub_model_id)

Resume from checkpoint: False
Project name: Qwen3-VL-8B-Instruct-with-codebook-GRPO-LoRA-v20260401131821
Hugging Face model ID: alxxtexxr/Qwen3-VL-8B-Instruct-with-codebook-GRPO-LoRA-v20260401131821


In [6]:
import os
from unsloth import FastVisionModel
import random
import numpy as np
import torch
import transformers

def set_seed(seed, verbose=True):
    # Set random seed for os
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':16:8'  # CUDA deterministic behavior (11.2+)

    # Set random seed for NumPy
    np.random.seed(seed)

    # Set random seed for Torch
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)           # If using multi-GPU
    # torch.use_deterministic_algorithms(True) # ERROR!
    torch.backends.cudnn.deterministic = True  # Ensures deterministic results
    torch.backends.cudnn.benchmark = False     # Avoids non-deterministic algorithms

    # Set random seed for Transformers
    transformers.set_seed(seed)

    # Set random seed for sklearn and Python's own random module
    random.seed(seed) 
    
    if verbose:
        print(f"Random seed: {seed}")

set_seed(SEED)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Random seed: 42


In [7]:
# Fix TorchDynamo issue
os.environ['TORCHDYNAMO_DISABLE'] = '1'
os.environ['UNSLOTH_DISABLE_COMPILE'] = '1'
os.environ['UNSLOTH_DISABLE_FUSED_KERNELS'] = '1'

# Fix Unsloth issue
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = ''

# Model

In [8]:
model, tokenizer = FastVisionModel.from_pretrained(
    model_name = SFT_LORA_ID,
    load_in_4bit = True, # Use 4bit to reduce memory use. False for 16-bit LoRA.
    use_gradient_checkpointing = 'unsloth', # True or 'unsloth' for long context
    dtype = torch.float16, # Force FP16
)

==((====))==  Unsloth 2026.3.18: Fast Qwen3_Vl patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

# Data

In [9]:
from datasets import load_dataset

dataset = load_dataset(RL_DATA_ID)
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['question', 'answer', 'PassRate_32BTrained', 'PassRate_7BBase', 'category', 'source', 'qid', 'image_paths', 'image'],
        num_rows: 1000
    })
    validation: Dataset({
        features: ['question', 'answer', 'PassRate_32BTrained', 'PassRate_7BBase', 'category', 'source', 'qid', 'image_paths', 'image'],
        num_rows: 125
    })
    test: Dataset({
        features: ['question', 'answer', 'PassRate_32BTrained', 'PassRate_7BBase', 'category', 'source', 'qid', 'image_paths', 'image'],
        num_rows: 125
    })
})


In [10]:
train_set = dataset['train']
val_set = dataset['validation']

print("Train set:")
print(train_set)
print("\nValidation set:")
print(val_set)

Train set:
Dataset({
    features: ['question', 'answer', 'PassRate_32BTrained', 'PassRate_7BBase', 'category', 'source', 'qid', 'image_paths', 'image'],
    num_rows: 1000
})

Validation set:
Dataset({
    features: ['question', 'answer', 'PassRate_32BTrained', 'PassRate_7BBase', 'category', 'source', 'qid', 'image_paths', 'image'],
    num_rows: 125
})


In [11]:
# Resize to (512, 512)
def resize_images(example):
    image = example["image"]
    image = image.resize((512, 512))
    example["decoded_image"] = image
    return example
train_set = train_set.map(resize_images)
val_set = val_set.map(resize_images)

# Then convert to RGB
def convert_to_rgb(example):
    image = example["decoded_image"]
    if image.mode != "RGB":
        image = image.convert("RGB")
    example["decoded_image"] = image
    return example
train_set = train_set.map(convert_to_rgb)
val_set = val_set.map(convert_to_rgb)

In [12]:
def clean_question(example):
    question = example["question"]
    question = question.replace("<image>", "")    # Remove "<image>" from the question text
    question = " ".join(question.strip().split()) # Remove extra whitespace
    example["question"] = question
    return example

train_set = train_set.map(clean_question)
val_set = val_set.map(clean_question)

In [14]:
# Define the delimiter variables for clarity and easy modification
REASONING_START = "<think>"
REASONING_END = "</think>"
SOLUTION_START = "<answer>"
SOLUTION_END = "</answer>"

def make_conversation(example):
    # Define placeholder constants if they are not defined globally
    # The user's text prompt
    text_content = (
        f"{example['question']}\n\n"
        "Please reason step by step "
        f"between {REASONING_START} and {REASONING_END}, "
        "and provide your final answer in LaTeX "
        "using \\boxed{} "
        f"between {SOLUTION_START} and {SOLUTION_END}"
    )

    # Construct the prompt in the desired multi-modal format
    prompt = [
        {
            "role": "user",
            "content": [
                {"type": "image"},  # Placeholder for the image
                {"type": "text", "text": text_content},  # The text part of the prompt
            ],
        },
    ]
    # The actual image data is kept separate for the processor
    return {"prompt": prompt, "image": example["decoded_image"], "answer": example["answer"]}

train_dataset = train_set.map(make_conversation)
val_dataset = val_set.map(make_conversation)

# We're reformatting dataset like this because decoded_images are the actual images
# The "image": example["decoded_image"] does not properly format the dataset correctly

# 1. Remove the original 'image' column
train_dataset = train_dataset.remove_columns("image")
val_dataset = val_dataset.remove_columns("image")

# 2. Rename 'decoded_image' to 'image'
train_dataset = train_dataset.rename_column("decoded_image", "image")
val_dataset = val_dataset.rename_column("decoded_image", "image")

In [18]:
image = train_dataset[100]["image"]
prompt = train_dataset[100]["prompt"]

inputs = tokenizer(
    image,
    prompt,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = False)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 1024,
                   use_cache = True, temperature = 1.0, min_p = 0.1)

<|im_start|>user
<|vision_start|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|ima

# Reward Functions

In [17]:
# Reward functions
import re

def formatting_reward_func(completions,**kwargs):
    import re
    thinking_pattern = f'{REASONING_START}(.*?){REASONING_END}'
    answer_pattern = f'{SOLUTION_START}(.*?){SOLUTION_END}'

    scores = []
    for completion in completions:
        if isinstance(completion, list):
            completion = completion[0]["content"] if completion else ""
        score = 0
        thinking_matches = re.findall(thinking_pattern, completion, re.DOTALL)
        answer_matches = re.findall(answer_pattern, completion, re.DOTALL)
        if len(thinking_matches) == 1:
            score += 1.0
        if len(answer_matches) == 1:
            score += 1.0

        # Fix up addCriterion issues
        # See https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl#qwen-2.5-vl-vision-rl-issues-and-quirks
        # Penalize on excessive addCriterion and newlines
        if len(completion) != 0:
            removal = completion.replace("addCriterion", "").replace("\n", "")
            if (len(completion)-len(removal))/len(completion) >= 0.5:
                score -= 2.0

        scores.append(score)
    return scores


def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    answer_pattern = f'{SOLUTION_START}(.*?){SOLUTION_END}'

    completions = [(c[0]["content"] if c else "") if isinstance(c, list) else c for c in completions]
    responses = [re.findall(answer_pattern, completion, re.DOTALL) for completion in completions]
    q = prompts[0]
    print('-'*20, f"Question:\n{q}", f"\nAnswer:\n{answer[0]}", f"\nResponse:{completions[0]}")
    return [
        2.0 if len(r)==1 and a == r[0].replace('\n','') else 0.0
        for r, a in zip(responses, answer)
    ]

# Training

In [19]:
from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    # Training arguments
    seed = SEED,
    
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    num_generations = 4, # Decrease if out of memory
    
    max_prompt_length = 1024,
    max_completion_length = 1024,
    
    # max_steps = 50,     # For testing
    num_train_epochs = 1, # Set this instead of max_steps for full training runs
    warmup_ratio = 0.05,
    learning_rate = 5e-6,
    lr_scheduler_type = "cosine",
    optim = "adamw_8bit",
    max_grad_norm = 0.1,
    weight_decay = 0.01,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    
    # Logging arguments
    logging_strategy='steps',
    logging_steps = 1,
    log_completions = False,
    report_to=['tensorboard', 'wandb'],
    
    # Saving arguments
    save_strategy='steps',
    # save_steps=1, # For testing
    save_steps=20,
    # save_total_limit=5, # 1 best + 4 recent checkpoints. Warning: It doesn't work
    
    # With load_best_model_at_end=True, your save_strategy will be ignored and default to eval_strategy.
    # So you will find one checkpoint at the end of each epoch.
    # https://discuss.huggingface.co/t/trainer-not-saving-after-save-steps/5464
    # load_best_model_at_end=True, 

    output_dir=project_name,
    hub_model_id=hub_model_id,
    push_to_hub=True,

    hub_strategy='all_checkpoints',
    hub_always_push=True,

    # Below enables GSPO:
    importance_sampling_level = "sequence",
    mask_truncated_completions = False,
    loss_type = "dr_grpo",
)

In [20]:
trainer = GRPOTrainer(
    model = model,
    args = training_args,
    # Pass the processor to handle multimodal inputs
    processing_class = tokenizer,
    reward_funcs = [
        formatting_reward_func,
        correctness_reward_func,
    ],
    train_dataset = train_dataset,
)

trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 1 | Total steps = 500
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 43,646,976 of 8,875,692,272 (0.49% trained)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: alimtegar to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/
`generation_config` default values have been modified to match model-specific defaults: {'temperature': 0.7, 'top_p': 0.8}. If this is not desired, please set these values explicitly.


-------------------- Question:
[{'content': [{'text': None, 'type': 'image'}, {'text': "In the figure, two equilateral triangles ABD and CBD each have side lengths of 1. Triangle ABD is translated to the right along the AC direction into position A'B'D', creating a shaded area. What is the perimeter of this shaded area? The above problem is with the following images:\n\nPlease reason step by step between <think> and </think>, and provide your final answer in LaTeX using \\boxed{} between <answer> and </answer>", 'type': 'text'}], 'role': 'user'}] 
Answer:
\boxed{2} 
Response:We are given two equilateral triangles, ABD and CBD, each with side length 1. The figure is symmetric, and point B is below, with A and C on the horizontal axis, and D above. The diagram shows that triangle ABD is translated to the right along the AC direction to form triangle A'B'D', and the shaded region is the intersection of the original triangle ABD and the translated triangle A'B'D'.

The shaded region is a r

Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / formatting_reward_func / mean,rewards / formatting_reward_func / std,rewards / correctness_reward_func / mean,rewards / correctness_reward_func / std
1,0.094000,0.375000,0.250000,831.500000,444.000000,1024.000000,0.625000,510.666687,444.000000,642.000000,0.028817,0.375000,0.517549,0.000000,0.000000
2,0.000000,1.000000,0.000000,319.125000,145.000000,608.000000,0.000000,319.125000,145.000000,608.000000,0.034418,1.000000,0.000000,0.000000,0.000000
3,0.124900,0.375000,0.538675,435.000000,218.000000,1024.000000,0.125000,350.857147,218.000000,530.000000,0.057139,0.375000,0.517549,0.000000,0.000000
4,0.044500,0.375000,0.250000,890.875000,475.000000,1024.000000,0.625000,669.000000,475.000000,940.000000,0.016665,0.375000,0.517549,0.000000,0.000000
5,-0.001000,0.750000,0.500000,759.875000,444.000000,1024.000000,0.500000,495.750000,444.000000,527.000000,0.018101,0.500000,0.534522,0.250000,0.707107
6,0.027300,1.250000,0.866025,658.750000,296.000000,1024.000000,0.250000,537.000000,296.000000,902.000000,0.040563,0.750000,0.462910,0.500000,0.925820
7,0.035700,0.250000,0.288675,802.750000,453.000000,1024.000000,0.500000,581.500000,453.000000,719.000000,0.019463,0.250000,0.462910,0.000000,0.000000
8,0.000000,0.000000,0.000000,1024.000000,1024.000000,1024.000000,1.000000,0.000000,0.000000,0.000000,0.024751,0.000000,0.000000,0.000000,0.000000
9,0.000000,1.000000,0.000000,612.375000,261.000000,883.000000,0.000000,612.375000,261.000000,883.000000,0.020217,1.000000,0.000000,0.000000,0.000000
10,0.017300,1.750000,0.288675,630.625000,243.000000,1024.000000,0.250000,499.500000,243.000000,989.000000,0.035489,0.750000,0.462910,1.000000,1.069045


-------------------- Question:
[{'content': [{'text': None, 'type': 'image'}, {'text': "On the 90th anniversary of the founding of the Communist Party of China, the County Education Bureau held a 'Red Songs Resound in China' activity. A 'Red Songs' singing competition was held in June. The organizing committee stipulates: any participant's score x satisfies: 60 ≤ x < 100. After the competition, the scores of all participants were organized as shown in the following table: According to the information provided in the table, n = ___.\n\nPlease reason step by step between <think> and </think>, and provide your final answer in LaTeX using \\boxed{} between <answer> and </answer>", 'type': 'text'}], 'role': 'user'}] 
Answer:
\boxed{0.25} 
Response:We are given a frequency distribution table of scores from a singing competition. The scores are grouped into intervals, and we are to find the value of $n$, which is the frequency for the interval $80 \le x < 90$.

The table is:

| 分数段 (Score Int

KeyboardInterrupt: 

# Inference

In [16]:
FastVisionModel.for_inference(model) # Enable for inference!

image = None
instruction = "Several slices of bread with side salads on a table."

messages = [
    {"role": "user", "content": [
        # {"type": "image"},
        {"type": "text", "text": instruction}
    ]}
]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
inputs = tokenizer(
    image,
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

In [17]:
from transformers import TextStreamer

text_streamer = TextStreamer(tokenizer, skip_prompt = False)
outputs = model.generate(**inputs, 
                   streamer = text_streamer, 
                   max_new_tokens = 512,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

<|im_start|>user
Several slices of bread with side salads on a table.<|im_end|>
<|im_start|>assistant
<|vtok_5424|><|vtok_1834|><|vtok_5261|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><